# Datamine ISOHOL Process Example

This notebook demonstrates how to configure and run the **`isohol`** process wrapper in `dmstudio`.

## Process Description

## ISOHOL

Create a plot file showing an isometric view of drillholes, from a user-defined viewpoint, described by rotation and elevation angles.

**ISOHOL** clarifies the relationship between drillholes and their ore intersections. Colour may be used for individual samples, if there is a field * **VALUE** in the input file containing the colour number for each sample. Each drillhole may be displayed with a bar chart, annotation or curve representing the values of fields in the file. Two displays per drillhole are possible; these may be two bar charts, two sets of annotation, two sets of curves, or one of each.

The input drillhole file is in standard sample format, as output for example by the [DESURV](<desurv.md>) process. The file must be sorted on **BHID** and **FROM**.

* **Note** (Scaling is fully automatic in this process.):

Process Prompts

This is an interactive process and prompts appear on the command line once it is launched:

## > HOLE ID LABELING; CHARACTER SIZE IN MM

> Enter required height; default is 4mm

## > ANGLE IN DEGREES CLOCKWISE FROM PROJECTED

## LINE OF FIRST SAMPLE IN EACH HOLE >

Supply angle; 180 is vertically upwards

## > HOW MANY FIELDS TO PLOT FOR EACH SAMPLE

(0, 1 or 2)?  >

Enter 0, 1 or 2.

## > SCALING FOR R.BAR OR ANNOTATION FIELD

> Enter field name

## > ANNOTATE (A) OR BARPLOT (B) OR NONE (N)?>

Enter A, B or N (the default is A)

If A is entered and the field is numeric

* **then**:

## > HOW MANY DECIMAL PLACES? >

Enter 0 - 4 (the default is 0)

If B is entered then:

## > MAX BAR LENGTH IN MM?

> Enter maximum bar length

## > DATA VALUE CORRESPONDING TO THIS LENGTH?

> Enter maximum data value.  Largest values
will be truncated and smaller ones scaled.

If 2 fields are to be plotted, the above
prompts are repeated for the second field.

If either one or two annotation responses
(A) given then:

## > CHARACTER SIZE IN MM? >

Enter required character size (default is
2 mm).

### Input Files:

* **in** (Drillhole):
  Input drillhole data file. Must contain fields BHID,X,Y,Z,LENGTH,A0,B0, + VALUE field.
  Required=Yes

* **proto** (Plot Prototype):
  Plot prototype file. Must contain the fields X, Y, S1, S2, CODE (numeric, explicit) and
  XMIN, XMAX, YMIN, YMAX, XSCALE, YSCALE (numeric, implicit). Without a plot prototype,
  PXMIN ,.... PZMAX define the region to be mapped to the screen.
  Required=No

### Output Files:

* **plot** (Plot):
  Output plot file.
  Required=Yes

### Fields:

* **value** (Numeric : IN):
  Field containing colour codes.
  Default=Undefined
  Required=No

* **bhid** (Any : IN):
  Drillhole identifier field (if not BHID).
  Default=BHID
  Required=No

* **x** (Numeric : IN):
  Name of X field (if not X).
  Default=X
  Required=No

* **y** (Numeric : IN):
  Name of Y field (if not Y).
  Default=Y
  Required=No

* **z** (Numeric : IN):
  Name of Z field (if not Z).
  Default=Z
  Required=No

* **length** (Numeric : IN):
  Name of LENGTH field (if not LENGTH).
  Default=LENGTH
  Required=No

* **a0** (Numeric : IN):
  Name of A0 field (if not A0).
  Default=A0
  Required=No

* **b0** (Numeric : IN):
  Name of B0 field (if not B0).
  Default=B0
  Required=No

### Parameters:

* **pxmin**:
  X value of left-hand side of region to be plotted.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **pxmax**:
  X value of right-hand side of region to be plotted.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **pymin**:
  Y value of front of region to be plotted.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **pymax**:
  Y value of back of region to be plotted.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **pzmin**:
  Z value of bottom of region to be plotted.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **pzmax**:
  Z value of top of region to be plotted.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **vertexag**:
  Vertical exaggeration required.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **rotate**:
  The rotation angle in degrees horizontally of the viewpoint, clockwise from the model Y
  axis (45).
  Range=0,360
  Values=Undefined
  Default=45
  Required=Yes

* **elevate**:
  The rotation angle in degrees vertically of the viewpoint, upwards from model X-Y plane
  (45).
  Range=-90,90
  Values=Undefined
  Default=45
  Required=Yes

* **charsize**:
  Character size in millimetres (3).
  Range=Undefined
  Values=Undefined
  Default=3
  Required=No

* **aspratio**:
  Aspect ratio, width / ht. for chars (0.9).
  Range=Undefined
  Values=Undefined
  Default=0.9
  Required=No

* **append**:
  Plot append flag. If set to 1 then the new plot will be appended to the PLOT file, if
  it exists and is a valid plot file (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('isohol')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute isohol
print("Running isohol...")
dm_cmd.isohol(
    in_i='t_assays',  # required
    proto_i='t_mod1',  # required
    plot_o='t_isohol_out',  # required
    length_f='optional',  # required
    a0_f='optional',  # required
    b0_f='optional',  # required
    pxmin_p='required_val',  # required
    pxmax_p='required_val',  # required
    pymin_p='required_val',  # required
    pymax_p='required_val',  # required
    pzmin_p='required_val',  # required
    pzmax_p='required_val',  # required
    vertexag_p='required_val',  # required
    rotate_p='required_val',  # required
    elevate_p='required_val',  # required
    # value_f='optional',  # optional
    # bhid_f='BHID',  # optional
    # x_f='X',  # optional
    # y_f='Y',  # optional
    # z_f='Z',  # optional
    # charsize_p=3,  # optional
    # aspratio_p=0.9,  # optional
    # append_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("isohol execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_isohol_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")